# CoRe-TFM: Reliability-Aware Extension Experiments

This notebook is the execution entry point for the new research direction: **inconsistency → reliability diagnosis → safe reconciliation decision**.

It deliberately separates (A) analyses that can be reproduced immediately from the archived bounded benchmark and (B) experiments requiring fresh TFM inference. Do not describe Section B results in the paper until those cells have actually been run and archived.

In [ ]:
from pathlib import Path
import json, sys, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
RESULTS = ROOT / 'results' / 'reliability_aware_v1'
RESULTS.mkdir(parents=True, exist_ok=True)
print('repo:', ROOT)

## A1. Run all archive-derived analyses
This produces oracle-opportunity/selection-regret decomposition, inconsistency-vs-gain correlations, model reliability proxies, candidate-family headroom, policy-transfer proxies, and analytic TV/accuracy counterexamples.

In [ ]:
cmd = [sys.executable, str(ROOT/'experiments'/'run_reliability_aware_suite.py'),
       '--fold-results', str(ROOT/'results'/'q1_fast_complete_256_v1'/'fold_results.csv'),
       '--output', str(RESULTS)]
subprocess.run(cmd, check=True)

## A2. Oracle opportunity vs. selection failure

In [ ]:
oracle = pd.read_csv(RESULTS/'oracle_selection_decomposition.csv')
display(oracle.groupby('model')[['available_opportunity','selection_regret','selective_minus_arithmetic']].mean())
display(oracle.groupby('model')['oracle_method'].value_counts().rename('count').to_frame())

fig, ax = plt.subplots(figsize=(7,4))
ax.scatter(oracle['available_opportunity'], oracle['selection_regret'], alpha=.65)
ax.set_xlabel('Available improvement over arithmetic')
ax.set_ylabel('Selection regret to oracle')
ax.set_title('Is Selective CoRe losing because there is no headroom, or because selection fails?')
plt.show()

## A3. Does inconsistency magnitude predict reconciliation benefit?

In [ ]:
ig = pd.read_csv(RESULTS/'inconsistency_vs_gain.csv')
print(json.loads((RESULTS/'inconsistency_vs_gain_correlations.json').read_text()))
fig, ax = plt.subplots(figsize=(7,4))
for model, g in ig.groupby('model'):
    ax.scatter(g.factorization_tv, g.selective_gain, alpha=.65, label=model)
ax.axhline(0, linewidth=1)
ax.set_xlabel('Factorization TV')
ax.set_ylabel('Arithmetic NLL - Selective NLL (positive = Selective helps)')
ax.legend()
plt.show()

## A4. Analytic counterexamples: coherence is not an accuracy metric

In [ ]:
from core_tfm.research_extensions import inconsistency_accuracy_counterexamples
cx = inconsistency_accuracy_counterexamples()
display(pd.DataFrame(cx['large_tv_no_need_to_repair']))
display(pd.DataFrame(cx['small_tv_repair_can_help']))

## A5. Diagnose TabICLv2 vs TabPFN-3 heterogeneity
The archived table contains raw-factorization and conditional scores but not every direct-marginal view metric. This cell gives the strongest valid proxy now; fresh reruns in Section B archive direct marginal NLL/Brier/ECE, per-class metrics, entropy, and per-example defects.

In [ ]:
rel = pd.read_csv(RESULTS/'model_view_reliability_proxy.csv')
display(rel.groupby('model').agg({
    'j1_joint_nll':'mean', 'j2_joint_nll':'mean',
    'j1_conditional_nll_a_given_b':'mean', 'j2_conditional_nll_b_given_a':'mean',
    'raw_direction_gap':'mean', 'factorization_tv':'mean'}))

## A6. Candidate-family complexity/headroom

In [ ]:
fam = pd.read_csv(RESULTS/'candidate_family_headroom.csv')
display(fam.groupby(['model','family'])['oracle_nll_within_family'].mean().unstack())

from core_tfm.research_extensions import complexity_penalty
for nv in [26,52,104,208,500]:
    print(nv, {k: round(complexity_penalty(nv,k),4) for k in [1,4,8,16,48]})

## A7. Inference-dispersion reliability mechanism (unit demonstration)
For actual TFM runs, stack predictions from admissible inference perturbations as `(members, examples, classes)`. First test whether dispersion predicts held-out error; only then use it for reconciliation weighting.

In [ ]:
from core_tfm.research_extensions import jensen_shannon_dispersion, inverse_dispersion_weight
rng = np.random.default_rng(42)
p1 = rng.dirichlet(np.ones(4), size=(8,100))
p2 = rng.dirichlet(np.ones(4)*4, size=(8,100))
d1 = jensen_shannon_dispersion(p1)
d2 = jensen_shannon_dispersion(p2)
w = inverse_dispersion_weight(d1,d2,temperature=.1)
pd.Series(w).describe()

## A8. Rare-class-aware penalty mechanism

In [ ]:
from core_tfm.research_extensions import support_adaptive_penalties
counts = np.array([1,2,5,10,25,100])
pd.DataFrame({'support':counts, 'lambda':support_adaptive_penalties(counts,base_lambda=10,tau=10)})

## A9. Known-truth downstream utility experiment template
Use this on controlled tasks where `P*` is known. It tests whether a probability method changes actual decision quality, not merely NLL.

In [ ]:
from core_tfm.research_extensions import decision_regret
rng = np.random.default_rng(7)
p_true = rng.dirichlet(np.ones(6), size=200).reshape(200,2,3)
q = .9*p_true + .1*(np.ones_like(p_true)/6)
utility = rng.normal(size=(5,2,3))
regret = decision_regret(q,p_true,utility)
print('mean decision regret:', regret.mean())

# B. Fresh-inference experiment matrix
Run these before manuscript submission. The canonical settings live in `configs/reliability_aware_experiments.yaml`.

1. **Five independent sampling seeds** × 5 folds × all 10 datasets.
2. **Context sizes** 64/128/256/512/1024 with at least 3 seeds.
3. **Third released TFM**, admitted only after operational preflight.
4. Archive direct-marginal and conditional NLL/Brier/ECE plus per-class reliability.
5. Archive per-example factorization TV, marginalization defects, entropy and inference dispersion.
6. Save all validation candidate scores to evaluate **Safe Selective CoRe** without test leakage.
7. Rare-class exclusion/support-adaptive penalty sensitivity.
8. Leave-one-dataset-out and model-global policy transfer.
9. Controlled known-truth downstream utility evaluation.

Multi-target graphical reconciliation, online reconciliation, conformalization and Wasserstein objectives remain explicit follow-up projects rather than being mixed into the main paper.

## Manuscript decision rule
The upgraded paper should make claims only after distinguishing: **available oracle opportunity**, **selection regret**, **view reliability**, and **validation complexity**. The intended story is not that reconciliation always wins; it is to identify when coherence repair is statistically justified.